# 01 Data Gathering

Notebook ini mengumpulkan semua dataset raw, menyamakan kolom minimum, memvalidasi struktur dasar, lalu menyimpan dataset gabungan ke `data/interim/combined_raw_transactions.csv`.

## Setup Environment

Tujuan: menyiapkan library, path project, path data raw/interim, dan opsi display agar proses gathering konsisten.

In [1]:
# Cell 1: Setup Environment
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re
from zipfile import ZipFile
import xml.etree.ElementTree as ET
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

try:
    from fingo_ds1.config import RAW_DATA_PATH, INTERIM_DATA_PATH
except ImportError:
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'
    INTERIM_DATA_PATH = PROJECT_ROOT / 'data' / 'interim'

RAW_DATA_PATH = Path(RAW_DATA_PATH)
INTERIM_DATA_PATH = Path(INTERIM_DATA_PATH)
INTERIM_DATA_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data path: {RAW_DATA_PATH}')
print(f'Interim data path: {INTERIM_DATA_PATH}')

Project root: /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Raw data path: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw
Interim data path: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim


## Load Household Transaction Data

Tujuan: membaca semua data household transaction dari folder raw, termasuk CSV dan Excel, lalu menggabungkannya menjadi satu dataframe household.

In [2]:
# Cell 2: Load Household Transaction Data
SUPPORTED_EXTENSIONS = {'.csv', '.xlsx', '.xls'}
XLSX_NS = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}


def excel_col_to_index(cell_ref):
    letters = ''.join(ch for ch in str(cell_ref) if ch.isalpha()) or 'A'
    index = 0
    for char in letters.upper():
        index = index * 26 + ord(char) - ord('A') + 1
    return index - 1


def read_xlsx_light(file_path):
    file_path = Path(file_path)
    with ZipFile(file_path) as archive:
        names = set(archive.namelist())
        shared_strings = []
        if 'xl/sharedStrings.xml' in names:
            shared_root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            for item in shared_root.findall('main:si', XLSX_NS):
                shared_strings.append(''.join(t.text or '' for t in item.findall('.//main:t', XLSX_NS)))

        sheet_path = 'xl/worksheets/sheet1.xml'
        sheet_root = ET.fromstring(archive.read(sheet_path))
        rows = []
        max_cols = 0

        for row in sheet_root.findall('.//main:sheetData/main:row', XLSX_NS):
            values = []
            for cell in row.findall('main:c', XLSX_NS):
                col_index = excel_col_to_index(cell.attrib.get('r', 'A1'))
                while len(values) <= col_index:
                    values.append(np.nan)

                cell_type = cell.attrib.get('t')
                if cell_type == 'inlineStr':
                    value = ''.join(t.text or '' for t in cell.findall('.//main:t', XLSX_NS))
                else:
                    raw_value = cell.find('main:v', XLSX_NS)
                    value = raw_value.text if raw_value is not None else np.nan
                    if cell_type == 's' and pd.notna(value):
                        value = shared_strings[int(value)]

                values[col_index] = value

            max_cols = max(max_cols, len(values))
            rows.append(values)

        if not rows:
            return pd.DataFrame()

        rows = [row + [np.nan] * (max_cols - len(row)) for row in rows]
        header = [f'column_{i}' if pd.isna(col) or col == '' else col for i, col in enumerate(rows[0])]
        return pd.DataFrame(rows[1:], columns=header)


def read_data_file(file_path):
    file_path = Path(file_path)
    if file_path.name.startswith('~$'):
        return None

    try:
        if file_path.suffix.lower() == '.csv':
            return pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')
        if file_path.suffix.lower() in {'.xlsx', '.xls'}:
            try:
                return pd.read_excel(file_path)
            except ImportError:
                return read_xlsx_light(file_path)
        return pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig')
    except Exception as exc:
        print(f'Error loading {file_path}: {exc}')
        return None


def iter_data_files(candidate):
    candidate = Path(candidate)
    if candidate.is_file():
        return [candidate]
    if candidate.is_dir():
        return sorted(
            path for path in candidate.rglob('*')
            if path.is_file()
            and path.suffix.lower() in SUPPORTED_EXTENSIONS
            and not path.name.startswith('~$')
        )
    print(f'Missing path skipped: {candidate}')
    return []


def load_dataset_group(candidates, source_name):
    frames = []
    loaded_files = 0

    for candidate in candidates:
        for file_path in iter_data_files(candidate):
            df = read_data_file(file_path)
            if df is None or df.empty:
                continue

            df = df.copy()
            df['_source_file'] = file_path.name
            df['_source_path'] = str(file_path.relative_to(PROJECT_ROOT)) if file_path.is_relative_to(PROJECT_ROOT) else str(file_path)
            frames.append(df)
            loaded_files += 1

    if not frames:
        print(f'No data loaded for {source_name}')
        return pd.DataFrame()

    print(f'{source_name}: {loaded_files} files loaded')
    return pd.concat(frames, ignore_index=True, sort=False)


def load_household_data():
    path = RAW_DATA_PATH / 'daily_household_transaction'
    candidates = [
        path / 'daily_household_transaction_clean',
        path / 'daily_household_transaction_raw_public',
        path / 'Daily_household_transactioncsv',
        path / 'Daily Household Transactions.csv',
    ]
    return load_dataset_group(candidates, 'household_transaction')


df_household = load_household_data()
print(f'Household data: {len(df_household):,} records')
display(df_household.head())

Missing path skipped: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/daily_household_transaction/Daily_household_transactioncsv
household_transaction: 49 files loaded
Household data: 49,567 records


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_source_file,_source_path,product_category,Jumlah,Returned quantity,Total Berat,Waktu Pengiriman Diatur,Status Pembatalan/ Pengembalian,No. Resi,Antar ke counter/ pick-up,Pesanan Harus Dikirimkan Sebelum (Menghindari keterlambatan),Waktu Pembayaran Dilakukan,SKU Induk,Nomor Referensi SKU,Nama Variasi,Harga Awal,Harga Setelah Diskon,Total Harga Produk,Diskon Dari Penjual,Diskon Dari Shopee,Berat Produk,Jumlah Produk di Pesan,Voucher Ditanggung Penjual,Cashback Koin,Voucher Ditanggung Shopee,Paket Diskon,Paket Diskon (Diskon dari Shopee),Paket Diskon (Diskon dari Penjual),Potongan Koin Shopee,Diskon Kartu Kredit,Ongkos Kirim Pengembalian Barang,Catatan dari Pembeli,Catatan,Username (Pembeli),Nama Penerima,No. Telepon,Alamat Pengiriman,Waktu Pesanan Selesai,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Load Ecommerce Sales Data

Tujuan: membaca seluruh file ecommerce sales clean dan raw public, lalu menggabungkannya menjadi satu dataframe ecommerce.

In [3]:
# Cell 3: Load Ecommerce Data
def load_ecommerce_data():
    path = RAW_DATA_PATH / 'Indonesian_Ecommerce_sales'
    candidates = [
        path / 'Indonesian_Ecommerce_sales_clean',
        path / 'Indonesian_Ecommerce_sales_raw_public',
    ]
    return load_dataset_group(candidates, 'ecommerce_sales')


df_ecommerce = load_ecommerce_data()
print(f'Ecommerce data: {len(df_ecommerce):,} records')
display(df_ecommerce.head())

ecommerce_sales: 48 files loaded
Ecommerce data: 47,106 records


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_source_file,_source_path,product_category,Jumlah,Returned quantity,Total Berat,Waktu Pengiriman Diatur,Status Pembatalan/ Pengembalian,No. Resi,Antar ke counter/ pick-up,Pesanan Harus Dikirimkan Sebelum (Menghindari keterlambatan),Waktu Pembayaran Dilakukan,SKU Induk,Nomor Referensi SKU,Nama Variasi,Harga Awal,Harga Setelah Diskon,Total Harga Produk,Diskon Dari Penjual,Diskon Dari Shopee,Berat Produk,Jumlah Produk di Pesan,Voucher Ditanggung Penjual,Cashback Koin,Voucher Ditanggung Shopee,Paket Diskon,Paket Diskon (Diskon dari Shopee),Paket Diskon (Diskon dari Penjual),Potongan Koin Shopee,Diskon Kartu Kredit,Ongkos Kirim Pengembalian Barang,Catatan dari Pembeli,Catatan,Username (Pembeli),Nama Penerima,No. Telepon,Alamat Pengiriman,Waktu Pesanan Selesai
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/Indonesian...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Load Monthly and Personal Finance Data

Tujuan: membaca dataset bulanan dan personal finance sebagai sumber tambahan untuk analisis impulsive buying behavior.

In [4]:
# Cell 4: Load Monthly and Personal Finance Data
def load_first_existing(candidates, source_name):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            df = read_data_file(candidate)
            if df is not None:
                print(f'{source_name} loaded from: {candidate}')
                return df
    print(f'No data loaded for {source_name}')
    return pd.DataFrame()


df_monthly = load_first_existing([
    RAW_DATA_PATH / 'all_months_clean.csv',
    RAW_DATA_PATH / 'Indonesian_Ecommerce_sales' / 'all_months_clean.csv',
    PROJECT_ROOT / 'data' / 'all_months' / 'all_months_clean.csv',
], 'monthly_clean')
print(f'Monthly data: {len(df_monthly):,} records')

df_personal = load_first_existing([
    RAW_DATA_PATH / 'personal_finance' / 'Personal_Finance_Dataset.csv',
], 'personal_finance')
print(f'Personal finance data: {len(df_personal):,} records')

display(df_monthly.head())

monthly_clean loaded from: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/Indonesian_Ecommerce_sales/all_months_clean.csv
Monthly data: 20,848 records
personal_finance loaded from: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw/personal_finance/Personal_Finance_Dataset.csv
Personal finance data: 1,500 records


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024.xlsx
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024.xlsx
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024.xlsx
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024.xlsx
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024.xlsx


## Inspect Column Structure

Tujuan: melihat nama kolom dan ukuran tiap dataset sebelum standardisasi, supaya mapping kolom bisa dicek jelas.

In [5]:
# Cell 5: Inspect Column Structure
datasets = {
    'household': df_household,
    'ecommerce': df_ecommerce,
    'monthly': df_monthly,
    'personal': df_personal,
}

for name, df in datasets.items():
    print(f'\n{name.upper()} columns:')
    print(df.columns.tolist())
    print(f'Shape: {df.shape}')


HOUSEHOLD columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', '_source_file', '_source_path', 'product_category', 'Jumlah', 'Returned quantity', 'Total Berat', 'Waktu Pengiriman Diatur', 'Status Pembatalan/ Pengembalian', 'No. Resi', 'Antar ke counter/ pick-up', 'Pesanan Harus Dikirimkan Sebelum (Menghindari keterlambatan)', 'Waktu Pembayaran Dilakukan', 'SKU Induk', 'Nomor Referensi SKU', 'Nama Variasi', 'Harga Awal', 'Harga Setelah Diskon', 'Total Harga Produk', 'Diskon Dari Penjual', 'Diskon Dari Shopee', 'Berat Produk', 'Jumlah Produk di Pesan', 'Voucher Ditanggung Penjual', 'Cashback Koin', 'Voucher Ditanggung Shopee', 'Paket Diskon', 'Paket Di

## Standardize Column Names

Tujuan: menyeragamkan format nama kolom dan menambahkan `data_source` agar asal data setiap record tetap terlacak.

In [6]:
# Cell 6: Standardize Column Names
def standardize_columns(df, source_name):
    df = df.copy()
    if df.empty:
        df['data_source'] = source_name
        return df

    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.replace(r'^\ufeff', '', regex=True)
        .str.lower()
        .str.strip()
        .str.replace(r'[^0-9a-zA-Z]+', '_', regex=True)
        .str.strip('_')
    )
    df['data_source'] = source_name
    return df


df_household = standardize_columns(df_household, 'household_transaction')
df_ecommerce = standardize_columns(df_ecommerce, 'ecommerce_sales')
df_monthly = standardize_columns(df_monthly, 'monthly_clean')
df_personal = standardize_columns(df_personal, 'personal_finance')

print('Columns standardized')

Columns standardized


## Identify Common Schema

Tujuan: mengumpulkan semua nama kolom unik dan menentukan kolom wajib untuk dataset gabungan.

In [7]:
# Cell 7: Identify Common Schema
all_columns = set()
for df in [df_household, df_ecommerce, df_monthly, df_personal]:
    all_columns.update(df.columns)

print(f'Total unique columns: {len(all_columns)}')
print(f'\nAll columns:\n{sorted(all_columns)}')

required_cols = ['date', 'amount', 'category', 'description']
print(f'\nRequired columns for alignment: {required_cols}')

Total unique columns: 67

All columns:
['alamat_pengiriman', 'alasan_pembatalan', 'amount', 'antar_ke_counter_pick_up', 'berat_produk', 'cashback_koin', 'catatan', 'catatan_dari_pembeli', 'category', 'currency', 'data_source', 'date', 'diskon_dari_penjual', 'diskon_dari_shopee', 'diskon_kartu_kredit', 'estimasi_potongan_biaya_pengiriman', 'harga_awal', 'harga_setelah_diskon', 'income_expense', 'jumlah', 'jumlah_produk_di_pesan', 'kota_kabupaten', 'metode_pembayaran', 'mode', 'nama_penerima', 'nama_variasi', 'no_resi', 'no_telepon', 'nomor_referensi_sku', 'note', 'num_product_categories', 'ongkos_kirim_dibayar_oleh_pembeli', 'ongkos_kirim_pengembalian_barang', 'opsi_pengiriman', 'order_id', 'paket_diskon', 'paket_diskon_diskon_dari_penjual', 'paket_diskon_diskon_dari_shopee', 'perkiraan_ongkos_kirim', 'pesanan_harus_dikirimkan_sebelum_menghindari_keterlambatan', 'potongan_koin_shopee', 'product_categories', 'product_category', 'provinsi', 'returned_quantity', 'sku_induk', 'source_file',

## Map Source Columns

Tujuan: memetakan nama kolom berbeda dari tiap sumber ke schema utama: `date`, `amount`, `category`, dan `description`.

In [8]:
# Cell 8: Map and Align Columns
def map_columns(df, source):
    df = df.copy()

    column_mapping = {
        'household_transaction': {
            'transaction_date': 'date',
            'total_amount': 'amount',
            'product_category': 'category',
            'product_categories': 'category',
            'product_name': 'description',
            'note': 'description',
            'waktu_pesanan_dibuat': 'date',
            'total_pembayaran': 'amount',
            'order_id': 'description',
        },
        'ecommerce_sales': {
            'order_date': 'date',
            'price': 'amount',
            'product_category': 'category',
            'product_categories': 'category',
            'product_name': 'description',
            'waktu_pesanan_dibuat': 'date',
            'total_pembayaran': 'amount',
            'order_id': 'description',
        },
        'monthly_clean': {
            'date': 'date',
            'amount': 'amount',
            'category': 'category',
            'description': 'description',
            'waktu_pesanan_dibuat': 'date',
            'total_pembayaran': 'amount',
            'product_categories': 'category',
            'order_id': 'description',
        },
        'personal_finance': {
            'date': 'date',
            'amount': 'amount',
            'category': 'category',
            'description': 'description',
            'transaction_description': 'description',
        },
    }

    mapping = column_mapping.get(source, {})
    for old_col, new_col in mapping.items():
        if old_col not in df.columns or old_col == new_col:
            continue
        if new_col not in df.columns:
            df = df.rename(columns={old_col: new_col})
        else:
            df[new_col] = df[new_col].where(df[new_col].notna(), df[old_col])

    return df.loc[:, ~df.columns.duplicated()]


df_household = map_columns(df_household, 'household_transaction')
df_ecommerce = map_columns(df_ecommerce, 'ecommerce_sales')
df_monthly = map_columns(df_monthly, 'monthly_clean')
df_personal = map_columns(df_personal, 'personal_finance')

print('Columns mapped')

Columns mapped


## Create Unified Schema

Tujuan: memastikan setiap dataframe punya kolom wajib dan tetap mempertahankan kolom tambahan untuk analisis lanjutan.

In [9]:
# Cell 9: Create Unified Schema
def align_schema(df, required_cols):
    df = df.copy()

    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan

    if 'data_source' not in df.columns:
        df['data_source'] = 'unknown'

    extra_cols = [col for col in df.columns if col not in required_cols and col != 'data_source']
    return df[required_cols + ['data_source'] + extra_cols]


df_household_aligned = align_schema(df_household, required_cols)
df_ecommerce_aligned = align_schema(df_ecommerce, required_cols)
df_monthly_aligned = align_schema(df_monthly, required_cols)
df_personal_aligned = align_schema(df_personal, required_cols)

print('Schema aligned')
print(f'\nHousehold: {df_household_aligned.shape}')
print(f'Ecommerce: {df_ecommerce_aligned.shape}')
print(f'Monthly: {df_monthly_aligned.shape}')
print(f'Personal: {df_personal_aligned.shape}')

Schema aligned

Household: (49567, 65)
Ecommerce: (47106, 57)
Monthly: (20848, 20)
Personal: (1500, 6)


## Combine All Datasets

Tujuan: menggabungkan semua dataframe yang sudah aligned menjadi satu dataset raw gabungan.

In [10]:
# Cell 10: Combine All Datasets
frames = [
    df_household_aligned,
    df_ecommerce_aligned,
    df_monthly_aligned,
    df_personal_aligned,
]
frames = [df for df in frames if not df.empty]

df_combined = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

print(f'Combined dataset shape: {df_combined.shape}')
print('\nRecords per source:')
if 'data_source' in df_combined.columns:
    print(df_combined['data_source'].value_counts())

Combined dataset shape: (119021, 66)

Records per source:
data_source
household_transaction    49567
ecommerce_sales          47106
monthly_clean            20848
personal_finance          1500
Name: count, dtype: int64


## Basic Validation

Tujuan: mengecek jumlah record, duplikasi, missing values, missing kolom critical, dan tipe data awal.

In [11]:
# Cell 11: Basic Validation
print('=== DATA VALIDATION ===\n')

print(f'Total records: {len(df_combined):,}')
print(f'Total columns: {len(df_combined.columns)}')

print('\nDuplicate rows:')
print(f'{df_combined.duplicated().sum():,} duplicate records')

if 'data_source' in df_combined.columns:
    print('\nDuplicate rows by source:')
    print(df_combined.groupby('data_source', dropna=False).apply(lambda x: x.duplicated().sum()))

print('\nMissing values per column:')
missing = df_combined.isnull().sum()
missing_pct = (missing / len(df_combined) * 100).round(2) if len(df_combined) else missing
missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct,
}).sort_values('missing_pct', ascending=False)
print(missing_df[missing_df['missing_count'] > 0])

print('\nCritical column missing percentage:')
critical_missing = missing_df.loc[[col for col in required_cols + ['data_source'] if col in missing_df.index]]
print(critical_missing)

print('\nData types:')
print(df_combined.dtypes)

=== DATA VALIDATION ===

Total records: 119,021
Total columns: 66

Duplicate rows:


4,933 duplicate records

Duplicate rows by source:


data_source
ecommerce_sales          2462
household_transaction    2471
monthly_clean               0
personal_finance            0
dtype: int64

Missing values per column:
                                                    missing_count  missing_pct
type                                                       117521        98.74
subcategory                                                117195        98.47
currency                                                   116560        97.93
mode                                                       116560        97.93
income_expense                                             116560        97.93
waktu_pesanan_selesai                                      116081        97.53
ongkos_kirim_pengembalian_barang                           116081        97.53
pesanan_harus_dikirimkan_sebelum_menghindari_ke...         116081        97.53
no_resi                                                    116081        97.53
antar_ke_counter_pick_up             

/tmp/ipykernel_66954/794144615.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df_combined.groupby('data_source', dropna=False).apply(lambda x: x.duplicated().sum()))


## Date Range Validation

Tujuan: mengubah kolom tanggal ke format datetime dan mengecek rentang tanggal serta jumlah tanggal invalid.

In [12]:
# Cell 12: Date Range Validation
def parse_mixed_dates(series):
    values = series.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    try:
        return pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)
    except TypeError:
        parsed = pd.to_datetime(values, errors='coerce')
        missing = parsed.isna() & values.notna()
        parsed.loc[missing] = pd.to_datetime(values[missing], errors='coerce', dayfirst=True)
        return parsed


df_combined['date'] = parse_mixed_dates(df_combined['date'])
valid_dates = df_combined['date'].dropna()

print('Date range validation:')
if valid_dates.empty:
    print('No valid dates found')
else:
    print(f'Min date: {valid_dates.min()}')
    print(f'Max date: {valid_dates.max()}')
    print(f'Date range: {(valid_dates.max() - valid_dates.min()).days} days')

print('\nRecords with invalid dates:')
print(f'{df_combined["date"].isnull().sum():,} records')

Date range validation:
Min date: 2015-01-01 00:00:00
Max date: 2025-11-30 23:17:00
Date range: 3986 days

Records with invalid dates:
10,808 records


## Amount Validation

Tujuan: mengubah amount ke numeric dan mengecek statistik nilai, null, negatif, serta nol.

In [13]:
# Cell 13: Amount Validation
amount_text = df_combined['amount'].astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
amount_text = amount_text.replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
df_combined['amount'] = pd.to_numeric(amount_text, errors='coerce')

print('Amount statistics:')
print(df_combined['amount'].describe())

print('\nRecords with invalid amounts:')
print(f'Null: {df_combined["amount"].isnull().sum():,}')
print(f'Negative: {(df_combined["amount"] < 0).sum():,}')
print(f'Zero: {(df_combined["amount"] == 0).sum():,}')

Amount statistics:
count    1.187130e+05
mean     2.679710e+04
std      1.075454e+05
min      0.000000e+00
25%      1.961600e+01
50%      1.441980e+02
75%      2.280000e+04
max      3.403591e+06
Name: amount, dtype: float64

Records with invalid amounts:
Null: 308
Negative: 0
Zero: 15,876


## Preview Final Dataset

Tujuan: melihat struktur akhir dataset gabungan dan sample record sebelum disimpan.

In [14]:
# Cell 14: Preview Final Dataset
print('=== FINAL DATASET PREVIEW ===\n')
print(df_combined.info())
print('\nFirst 10 records:')
display(df_combined.head(10))

=== FINAL DATASET PREVIEW ===



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119021 entries, 0 to 119020
Data columns (total 66 columns):
 #   Column                                                      Non-Null Count   Dtype         
---  ------                                                      --------------   -----         
 0   date                                                        108213 non-null  datetime64[ns]
 1   amount                                                      118713 non-null  float64       
 2   category                                                    119021 non-null  object        
 3   description                                                 118500 non-null  object        
 4   data_source                                                 119021 non-null  object        
 5   order_id                                                    47106 non-null   object        
 6   total_qty                                                   62544 non-null   object        
 7   total_weigh

,date,amount,category,description,data_source,order_id,total_qty,total_weight_gr,total_returned_qty,total_diskon,product_categories,num_product_categories,status_pesanan,alasan_pembatalan,opsi_pengiriman,metode_pembayaran,kota_kabupaten,provinsi,ongkos_kirim_dibayar_oleh_pembeli,estimasi_potongan_biaya_pengiriman,total_pembayaran,perkiraan_ongkos_kirim,waktu_pesanan_dibuat,source_file,source_path,product_category,jumlah,returned_quantity,total_berat,waktu_pengiriman_diatur,status_pembatalan_pengembalian,no_resi,antar_ke_counter_pick_up,pesanan_harus_dikirimkan_sebelum_menghindari_keterlambatan,waktu_pembayaran_dilakukan,sku_induk,nomor_referensi_sku,nama_variasi,harga_awal,harga_setelah_diskon,total_harga_produk,diskon_dari_penjual,diskon_dari_shopee,berat_produk,jumlah_produk_di_pesan,voucher_ditanggung_penjual,cashback_koin,voucher_ditanggung_shopee,paket_diskon,paket_diskon_diskon_dari_shopee,paket_diskon_diskon_dari_penjual,potongan_koin_shopee,diskon_kartu_kredit,ongkos_kirim_pengembalian_barang,catatan_dari_pembeli,catatan,username_pembeli,nama_penerima,no_telepon,alamat_pengiriman,waktu_pesanan_selesai,mode,subcategory,income_expense,currency,type
0,2024-04-01 00:15:00,38300.0,Celengan,ORD_0000001,household_transaction,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-04-01 01:47:00,18576.0,Celengan,ORD_0000002,household_transaction,ORD_0000002,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-04-01 04:25:00,7069.0,Celengan,ORD_0000003,household_transaction,ORD_0000003,1,500,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-04-01 04:41:00,32200.0,Mangkok Sambal / Saus,ORD_0000004,household_transaction,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-04-01 06:12:00,0.0,"Keranjang, Other, Tempat Nasi",ORD_0000005,household_transaction,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-04-01 07:09:00,40996.0,Celengan,ORD_0000006,household_transaction,ORD_0000006,2,1000,0,0,Celengan,1,Selesai,,Hemat Kargo-SPX Hemat,SPayLater,KAB. BANDUNG,JAWA BARAT,0,10000,40996,10000,2024-04-01 07:09,AprilSales2024_clean.xlsx,data/raw/daily_household_transaction/daily_hou...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

## Save Combined Dataset

Tujuan: menyimpan dataset gabungan ke folder interim sebagai `combined_raw_transactions.csv`.

In [15]:
# Cell 15: Save Combined Dataset
output_path = INTERIM_DATA_PATH / 'combined_raw_transactions.csv'
INTERIM_DATA_PATH.mkdir(parents=True, exist_ok=True)
df_combined.to_csv(output_path, index=False)

print(f'Dataset saved to: {output_path}')
print(f'File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB')

Dataset saved to: /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/interim/combined_raw_transactions.csv
File size: 40.98 MB
